# Modelo Híbrido DenseNet121 + MLP


## Objetivo do Script_09 no Escopo Silver KNN

O script_09, em sua versão original, itera sobre os 5 datasets treinando um modelo por dataset. Na configuração
Silver KNN deste roteiro, o loop é simplificado para treinar apenas 1 modelo — o SILVER_KNN — avaliado no test
set canônico do GOLD. Isso reduz o tempo total de execução em aproximadamente 80%.
5.3 Adaptação do CONFIG para Rodar Apenas Silver KNN

```python
# Em CONFIG['datasets'] (script_09_modelo_hibrido.py):
# Manter apenas o SILVER_KNN no dicionário:
'datasets': {
'GOLD' : {'file': '../csv/ecg_gold_completo_classified.csv',
'n': 3013, 'imputacao_pct': 0.0, 'metodo': 'Sem imputacao'},
'SILVER_KNN': {'file': '../csv/ecg_silver_knn_imputado_classified.csv',
'n': 3481, 'imputacao_pct': 1.67, 'metodo': 'KNN'},
# Remover ou comentar BRONZE_HYBRID, BRONZE_MICE, BRONZE_KNN
},
# ATENÇÃO: GOLD deve permanecer no dicionário pois gerar_analise_comparativa()
# faz referência a todos_metricas['GOLD'] para calcular Delta_GOLD.
# Ou adapte a função para calcular delta em relação ao próprio SILVER_KNN.
```

> Alternativa simplificada:
> 
> Se o objetivo é apenas treinar e avaliar o SILVER KNN sem comparação com GOLD, pode-se também remover o GOLD do dicionário e simplificar a função gerar_analise_comparativa() para não calcular Delta_GOLD. Nesse caso, os resultados serão apresentados apenas para o SILVER KNN.


## Arquitetura do Modelo Híbrido

A arquitetura HybridECGClassifier integra dois branches especializados fundidos por concatenação:

| Componente | Input → Output                 | Descrição                                                                                                   |
| ---------- | ------------------------------ | ----------------------------------------------------------------------------------------------------------- |
| Branch CNN | `[B, 1, 272, 512] → [B, 1024]` | DenseNet121 pré-treinado em ImageNet, com `conv0` adaptado para 1 canal usando a média dos pesos originais. |
| Branch MLP | `[B, 14] → [B, 32]`            | Rede densa composta por `Linear(14→64)` + `ReLU` + `Dropout(0.4)` + `Linear(64→32)`.                        |
| Fusão      | `[B, 1056] → [B, 2]`           | `CONCAT([1024, 32])` seguido de `FC(1056→256→128→2)` + `Dropout` + `Softmax`.                               |




## Estratégia de Treinamento em 2 Fases

### Fase 1 — Warm-up (10 épocas, CNN congelada)

- CNN (DenseNet121) completamente congelada — apenas MLP e camadas de fusão treinadas
- LR único: 1e-3 (otimizador Adam)
- Objetivo: inicializar os pesos da fusão sem destruir representações ImageNet da CNN


### Fase 2 — Fine-tuning (40 épocas, rede completa)

- Todos os parâmetros liberados para treinamento
- LRs diferenciados: CNN: 1e-5 | MLP: 1e-4 | Fusão: 1e-4
- ReduceLROnPlateau: fator 0.5, paciência 5 épocas no val_loss
- Early stopping: paciência 10 épocas no val_AUC
- Gradient clipping: grad_clip=1.0 para estabilidade numérica
- Class weights: calculados sobre y_train do SILVER KNN via compute_class_weight


## Métricas de Avaliação

O modelo é avaliado no test set canônico do GOLD (452 registros). As métricas calculadas são:
- AUC-ROC: principal métrica — meta >= 0,95
- Accuracy: acurácia geral no test set
- F1 Macro: média dos F1 de ambas as classes
- F1 NORMAL (classe 0) e F1 ANORMAL (classe 1): análise por classe
- Baseline de referência: XGBoost tabular puro — AUC = 0,928


## Execução

- Garantir que o test set canônico do GOLD está disponível
```bash
ls splits/test_indices.npy # deve existir e conter 452 índices GOLD
```

> Tempo estimado: 30–90 minutos (CPU) | 10–25 minutos (GPU NVIDIA com CUDA). Com SILVER KNN e apenas
> 1 modelo (sem os 4 datasets adicionais), o tempo é reduzido ~80% em relação ao script original.


## Artefatos Gerados

| Artefato                     | Local                    | Conteúdo                                                                   |
| ---------------------------- | ------------------------ | -------------------------------------------------------------------------- |
| `modelo_SILVER_KNN_final.pt` | `checkpoints/`           | Pesos finais do modelo, métricas, configurações e scaler.                  |
| `historico_SILVER_KNN.json`  | `resultados_e_metricas/` | Histórico de treino contendo loss e AUC por época (warm-up + fine-tuning). |
| `comparative_results.csv`    | `resultados_e_metricas/` | Tabela comparativa com métricas do(s) dataset(s) treinado(s).              |
| `comparative_roc.png`        | `plots_comparativos/`    | Curva ROC do modelo SILVER KNN comparada ao baseline XGBoost.              |
| `comparative_auc_barras.png` | `plots_comparativos/`    | Gráfico de barras da AUC-ROC com linha de meta em `0.95`.                  |
| `script_09_MAIN_*.log`       | `resultados_e_metricas/` | Log completo de execução contendo timestamps e métricas por época.         |


# Config

In [42]:
import os
import sys
import time
import json
import logging
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import models
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    classification_report, roc_curve
)

from ipynb.fs.full.script_08_pytorch_dataset import (
    ECGMultimodalDataset, get_train_transform, get_eval_transform
)

CONFIG = {
    # Paths
    'image_dir': '../image_tracings',
    'splits_dir': './splits',
    'results_dir': '../resultados_e_metricas/script_08_pytorch_dataset',
    'checkpoints_dir': '../checkpoints',

    # Cinco datasets do estudo comparativo
    'datasets': {
        'GOLD': {'file': '../csv/ecg_gold_completo_classified.csv', 'n': 3013, 'imputacao_pct': 0.0,   'metodo': 'Sem imputacao'},
        'SILVER_KNN': {'file': '../csv/ecg_silver_knn_imputado_classified.csv', 'n': 3481, 'imputacao_pct': 1.67,  'metodo': 'KNN'},
        # 'BRONZE_HYBRID': {'file': '../csv/ecg_bronze_hybrid_imputado_classified.csv', 'n': 3664, 'imputacao_pct': 3.0,   'metodo': 'Hibrido KNN+MICE'},
        # 'BRONZE_MICE': {'file': '../csv/ecg_bronze_mice_imputado_classified.csv', 'n': 3664, 'imputacao_pct': 3.0,   'metodo': 'MICE'},
        # 'BRONZE_KNN': {'file': '../csv/ecg_bronze_knn_imputado_classified.csv', 'n': 3664, 'imputacao_pct': 3.0,   'metodo': 'KNN'},
    },

    # Colunas
    'param_cols': [
        'HR', 'Pd', 'PR', 'QRS_Dur', 'QT', 'QTC',
        'P_axis', 'QRS_axis', 'T_axis',
        'RV5', 'SV1', 'RV5_SV1_sum', 'RV6', 'SV2'
    ],
    'label_col'    : 'classificacao',
    'filename_col' : 'filename',

    # Imagem]
    'img_resize'    : (272, 512),
    'img_channels'  : 1,
    'normalize_mean': [0.5],
    'normalize_std' : [0.5],

    # Data augmentation
    'aug_translate'  : 0.02,
    'aug_scale_min'  : 0.98,
    'aug_scale_max'  : 1.02,
    'aug_brightness' : 0.1,
    'aug_contrast'   : 0.1,

    # Arquitetura
    'cnn_features' : 1024,
    'mlp_out'      : 32,
    'fusion_dim'   : 1056,
    'n_classes'    : 2,
    'dropout'      : 0.4,

    # Treinamento
    'batch_size'  : 32,
    'num_workers' : 0,
    'pin_memory'  : False,
    'random_seed' : 42,

    # Fase 1 — Warm-up (CNN congelada)
    'warmup_epochs' : 10,
    'warmup_lr'     : 1e-3,

    # Fase 2 — Fine-tuning (rede completa, LRs diferenciados)
    'finetune_epochs' : 40,
    'lr_cnn'          : 1e-5,
    'lr_mlp'          : 1e-4,
    'lr_fusion'       : 1e-4,

    # Regularizacao e controle
    'weight_decay'        : 1e-4,
    'lr_patience'         : 5,
    'lr_factor'           : 0.5,
    'early_stop_patience' : 10,
    'grad_clip'           : 1.0,

    # Limiar H0: Delta AUC-ROC < 2% => robusto
    'delta_threshold' : 0.02,
}

# Configure logging

In [43]:
def configurar_logging(results_dir, tag='MAIN'):
    os.makedirs(results_dir, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_path = os.path.join(results_dir, f'script_09_{tag}_{ts}.log')
    logger = logging.getLogger(f'rescue2_e7_{tag}')
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        fmt = logging.Formatter('%(asctime)s  %(levelname)-8s  %(message)s',
                                datefmt='%H:%M:%S')
        fh = logging.FileHandler(log_path, encoding='utf-8')
        ch = logging.StreamHandler(sys.stdout)
        fh.setFormatter(fmt); ch.setFormatter(fmt)
        logger.addHandler(fh); logger.addHandler(ch)
    return logger

# Hybrid model -> DenseNet121/ResNet50/EfficientNet_B0 + MLP

In [44]:
class HybridECGClassifier(nn.Module):
    def __init__(self, config, backbone):
        super().__init__()
        dropout = config['dropout']
        n_tab = len(config['param_cols'])
        self.backbone = backbone

        if self.backbone == 'densenet121':
            # Branch CNN: DenseNet121 pre-treinado com conv0 adaptado 3ch -> 1ch
            densenet = models.densenet121(weights='IMAGENET1K_V1')
            n_cnn    = densenet.classifier.in_features  # 1024

            # Inicializacao correta: media dos pesos ImageNet RGB preserva magnitude
            old_w = densenet.features.conv0.weight.data
            new_conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            new_conv0.weight.data = old_w.mean(dim=1, keepdim=True)
            densenet.features.conv0 = new_conv0
            densenet.classifier = nn.Identity()
            self.cnn_branch = densenet

        elif self.backbone == 'resnet50':
            resnet = models.resnet50(weights='IMAGENET1K_V1')
            n_cnn = resnet.fc.in_features  # 2048

            old_w = resnet.conv1.weight.data  # [64, 3, 7, 7]
            new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            new_conv.weight.data = old_w.mean(dim=1, keepdim=True)
            resnet.conv1 = new_conv
            resnet.fc = nn.Identity()
            self.cnn_branch = resnet

        elif self.backbone == 'efficientnet_b0':
            efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
            n_cnn = efficientnet.classifier[1].in_features # 1280

            old_w = efficientnet.features[0][0].weight.data
            new_conv = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
            new_conv.weight.data = old_w.mean(dim=1, keepdim=True)
            efficientnet.features[0][0] = new_conv
            efficientnet.classifier = nn.Identity()
            self.cnn_branch = efficientnet
        
        else:
            raise ValueError(f'Backbone {self.backbone} não identificado. Espera-se densenet121, resnet50 ou efficientnet_b0')
        
        # Branch MLP: 14 -> 128 -> 64 -> 32
        self.mlp_branch = nn.Sequential(
            nn.Linear(n_tab, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, 64),   nn.BatchNorm1d(64),  nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(64, 32),    nn.BatchNorm1d(32),  nn.ReLU(inplace=True),
        )

        # Fusao: [1056] -> [256] -> [128] -> [2]
        self.fusion = nn.Sequential(
            nn.Linear(n_cnn + 32, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, config['n_classes']),
        )

    def forward(self, image, params):
        return self.fusion(torch.cat([self.cnn_branch(image), self.mlp_branch(params)], dim=1))

    def congelar_cnn(self):
        for p in self.cnn_branch.parameters(): p.requires_grad = False

    def descongelar_cnn(self):
        for p in self.cnn_branch.parameters(): p.requires_grad = True

    def contar_parametros(self):
        total = sum(p.numel() for p in self.parameters())
        train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {'total': total, 'treinaveis': train, 'congelados': total - train}

# MLP Model

In [45]:
class MLPClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        dropout = config['dropout']
        n_tab = len(config['param_cols'])

        # Branch MLP: 14 -> 128 -> 64 -> 32 -> 2
        self.mlp_branch = nn.Sequential(
            nn.Linear(n_tab, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(32, 2)
        )
    
    def forward(self, params, image = None):
        return self.mlp_branch(params)

    def contar_parametros(self):
        total = sum(p.numel() for p in self.parameters())
        train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {'total': total, 'treinaveis': train, 'congelados': total - train}

# CNN Model -> Densenet-121

In [46]:
class CNNClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        dropout = config['dropout']

        # Branch CNN: DenseNet121 pre-treinado com conv0 adaptado 3ch -> 1ch
        densenet = models.densenet121(weights='IMAGENET1K_V1')
        n_cnn    = densenet.classifier.in_features  # 1024

        # Inicializacao correta: media dos pesos ImageNet RGB preserva magnitude
        old_w = densenet.features.conv0.weight.data
        new_conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        new_conv0.weight.data = old_w.mean(dim=1, keepdim=True)
        densenet.features.conv0 = new_conv0
        densenet.classifier = nn.Identity()
        self.cnn_branch = densenet

        self.classifier = nn.Sequential(
            nn.Linear(n_cnn, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def forward(self, image, params = None):
        features = self.cnn_branch(image)
        return self.classifier(features)

    def congelar_cnn(self):
        for p in self.cnn_branch.parameters(): p.requires_grad = False

    def descongelar_cnn(self):
        for p in self.cnn_branch.parameters(): p.requires_grad = True

    def contar_parametros(self):
        total = sum(p.numel() for p in self.parameters())
        train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {'total': total, 'treinaveis': train, 'congelados': total - train}

In [47]:
for model in ['densenet121', 'efficientnet_b0', 'resnet50']:
    model = HybridECGClassifier(CONFIG, backbone='densenet121')
    imgs   = torch.randn(2, 1, 272, 512)
    params = torch.randn(2, 14)
    out    = model(imgs, params)
    print(f'{type(model).__name__}: \t{out.shape}')

model = CNNClassifier(CONFIG)
imgs   = torch.randn(2, 1, 272, 512)
params = torch.randn(2, 14)
out    = model(imgs)
print(f'{type(model).__name__}: \t{out.shape}')  # esperado: torch.Size([2, 2])

model = MLPClassifier(CONFIG)
imgs   = torch.randn(2, 1, 272, 512)
params = torch.randn(2, 14)
out    = model(params)
print(f'{type(model).__name__}: \t{out.shape}')  # esperado: torch.Size([2, 2])



HybridECGClassifier: 	torch.Size([2, 2])
HybridECGClassifier: 	torch.Size([2, 2])
HybridECGClassifier: 	torch.Size([2, 2])
CNNClassifier: 	torch.Size([2, 2])
MLPClassifier: 	torch.Size([2, 2])
